<a href="https://colab.research.google.com/github/kdrhanegrilmez/mL-Notebooks/blob/main/penguins.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🐧 Problem Tanımı (Penguins Veri Seti)
1. İş Problemi Nedir?
Palmer Adası (Antarktika) üzerinde yapılan bir çalışmada, araştırmacılar 3 farklı penguen türü (Adelie, Chinstrap, Gentoo) üzerinde fiziksel ölçümler kaydetmiştir. Senin görevin; elindeki fiziksel ölçümlere (gaga boyu, yüzgeç uzunluğu, vücut ağırlığı vb.) ve yaşadığı ada ile cinsiyetine bakarak, yeni gözlemlenen bir penguenin hangi türe ait olduğunu tahmin eden bir sınıflandırma modeli geliştirmektir.

2. Veri Seti Özeti:

Boyut: Yaklaşık 344 örnek.

Hedef (y): species (Penguen türü - 3 sınıf: Adelie, Chinstrap, Gentoo).

Öznitelikler (X):

Sayısal: bill_length_mm (Gaga uzunluğu), bill_depth_mm (Gaga derinliği), flipper_length_mm (Yüzgeç uzunluğu), body_mass_g (Vücut ağırlığı), year (Yıl).

Kategorik (Metin): island (Ada ismi: Biscoe, Dream, Torgersen), sex (Cinsiyet: male, female).

Özel Durum (Zorluk): Bu veri setinde eksik değerler (NaN) vardır ve KNN mesafe tabanlı bir algoritma olduğu için tüm verilerin sayısal olması gerekir.

In [ ]:
#Colabın ilk hücresi: Kütüphaneleri içe aktarıyoruz
#Veri analizi ve matris işlemleri için Numpy ve Pandas

import numpy as np
import pandas as pd

#Modelleme ve Önişleme için Sklearn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report,confusion_matrix, accuracy_score

#Veri setini internetten indirmek için (Colab'a ait bir özellik)
import urllib.request

In [ ]:
# 2. Penguen veri setini indir ve yükle
url = 'https://raw.githubusercontent.com/allisonhorst/palmerpenguins/main/inst/extdata/penguins.csv'
# Pandas'ın read_csv fonksiyonu ile veriyi oku ve bir "DataFrame" (tablo) haline getir.
df=pd.read_csv(url)

#Sütun isminde gizli boşluk varsa bu şekilde o boşluklar temizlenir.
df.columns=df.columns.str.strip()

In [ ]:
print(df.isnull().sum())
print(df.info())

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
year                  0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    object 
 7   year               344 non-null    int64  
dtypes: float64(4), int64(1), object(3)
memory usage: 21.6+ KB
None


In [ ]:
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [ ]:
#Eksik sütünları silmek yerine sayısal verileri ortalama, cinsiyet gibi kategorik verileri mod ile dolduracağız
#df=df.dropna()
#print(df.isnull().sum())
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    object 
 7   year               344 non-null    int64  
dtypes: float64(4), int64(1), object(3)
memory usage: 21.6+ KB
None


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [ ]:
print(df.isnull().sum())

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
year                  0
dtype: int64


In [ ]:
# 1. Sayısal sütunları (body_mass_g, bill_length_mm, vs.) 'median' ile doldur
# (Çok uç değer olmadığı için medyan ortalamadan daha güvenlidir)
df['bill_length_mm'] = df['bill_length_mm'].fillna(df['bill_length_mm'].median())
df['bill_depth_mm'] = df['bill_depth_mm'].fillna(df['bill_depth_mm'].median())
df['flipper_length_mm'] = df['flipper_length_mm'].fillna(df['flipper_length_mm'].median())
df['body_mass_g'] = df['body_mass_g'].fillna(df['body_mass_g'].median())

# 2. Kategorik sütunu (sex) 'mode' (en sık görülen) ile doldur
df['sex'] = df['sex'].fillna(df['sex'].mode()[0])

In [ ]:
print(df.isnull().sum())
print(df.info())
df.head()

species              0
island               0
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
sex                  0
year                 0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm     344 non-null    float64
 3   bill_depth_mm      344 non-null    float64
 4   flipper_length_mm  344 non-null    float64
 5   body_mass_g        344 non-null    float64
 6   sex                344 non-null    object 
 7   year               344 non-null    int64  
dtypes: float64(4), int64(1), object(3)
memory usage: 21.6+ KB
None


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.10,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.50,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.30,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,44.45,17.3,197.0,4050.0,male,2007
4,Adelie,Torgersen,36.70,19.3,193.0,3450.0,female,2007


In [ ]:
# X ve y ayır
y = df['species']
x = df.drop('species', axis=1)

# Kategorik verileri kodla (Artık boşluk olmadığı için sorunsuz çalışacak)
x = pd.get_dummies(x, columns=['island', 'sex'], drop_first=True)

In [ ]:
# Veriyi %80 Eğitim, %20 Test olarak ayırıyoruz.
# random_state=42 demek, her çalıştırdığımızda aynı rastgele bölünmeyi al demek (tekrarlanabilirlik için).
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
print(f"Eğitim verisi boyutu:{x_train.shape}")
print(f"Test verisi boyutu:{x_test.shape}")



Eğitim verisi boyutu:(275, 8)
Test verisi boyutu:(69, 8)


In [ ]:
# Pipeline, bir işçi zinciridir. İş akışını tek bir pakette toplar.
# Burada diyoruz ki:
# 1. Önce veriyi Standardize et (StandartScaler)
# 2. Sonra KNN modelini uygula (KNeighborsClassifier)

pipeline=Pipeline([
    ('scaler',StandardScaler()),
    ('knn',KNeighborsClassifier())
])

# Hiperparametre Optimizasyonu (GridSearchCV):
# K değerini (n_neighbors) 1'den 30'a kadar tek tek dene ve hangisinin en iyi olduğunu bul.

param_grid={'knn__n_neighbors':range(1,31)}

# 5-Katlı Çapraz Doğrulama ile en iyi K değerini seç.
grid_search=GridSearchCV(pipeline,param_grid,cv=5)
grid_search.fit(x_train,y_train)

print("Bulunan en iyi K değeri:",grid_search.best_params_)
print("en iyi eğitim başarısı:",grid_search.best_score_)


Bulunan en iyi K değeri: {'knn__n_neighbors': 7}
en iyi eğitim başarısı: 0.989090909090909


In [ ]:
# Test verisi üzerinde tahmin yap (Modelin hiç görmediği veri)
y_predict=grid_search.predict(x_test)
print("Doğruluk (Accuracy) Skoru:", accuracy_score(y_test, y_predict))
print("\nKarmaşıklık Matrisi (Confusion Matrix):")
print(confusion_matrix(y_test,y_predict))
print("\nSınıflandırma Raporu:")
print(classification_report(y_test,y_predict))


Doğruluk (Accuracy) Skoru: 1.0

Karmaşıklık Matrisi (Confusion Matrix):
[[32  0  0]
 [ 0 13  0]
 [ 0  0 24]]

Sınıflandırma Raporu:
              precision    recall  f1-score   support

      Adelie       1.00      1.00      1.00        32
   Chinstrap       1.00      1.00      1.00        13
      Gentoo       1.00      1.00      1.00        24

    accuracy                           1.00        69
   macro avg       1.00      1.00      1.00        69
weighted avg       1.00      1.00      1.00        69

